In [1]:
# Ruta del directorio actual
import os; ruta_actual = os.getcwd() 
# os.startfile(ruta_actual)

# CICLO DE ACOPLAMIENTO (TRANSPORTE-QUIMICA) UTILIZANDO (GWF,MT3D,WMA)

# 1. Preparación del entorno, se genera "conc_i.m3d"

In [4]:
# Preparación del entorno
import os, sys
import numpy as np 
import flopy
import matplotlib.pyplot as plt
from flopy.utils.binaryfile import HeadFile, CellBudgetFile, UcnFile
from flopy.plot.styles import styles
import pandas as pd 

# Generando un arreglo sconc en conc_i.m3d
sconc = np.ones((1, 1, 31), dtype=float)
sconc[0,0,10]=200
# Escribe sconc en "conc_i.m3d"
#np.savetxt("conc_i.m3d", sconc.reshape(-1, sconc.shape[-1]), fmt="%.1f")
np.savetxt("u_tilde.dat", sconc.reshape(-1, sconc.shape[-1]), fmt="%.1f")
print("Iniciamos sconc = u_tilde.dat ",sconc)

Iniciamos sconc = u_tilde.dat  [[[  1.   1.   1.   1.   1.   1.   1.   1.   1.   1. 200.   1.   1.   1.
     1.   1.   1.   1.   1.   1.   1.   1.   1.   1.   1.   1.   1.   1.
     1.   1.   1.]]]


# Construcción del programa RT.F90

# Ejecución de RT.EXE

# 3. Se ejecuta main_interfaz_local.exe para generar u_new.out

In [8]:
# COMPILAMOS Y EJECUTAMOS "RT"

# AL EJECUTAR RT.EXE, Se lee conc_i.m3d y s genera conc_s.txt

!main_interfaz_local.exe

 Esta es la interfaz para resolver iteraciones de mezcla reactiva en 1D con el WMA usando Euler explicito.
 Directorio de las bases de datos: 
 Directorio del problema: 
 Root de los archivos de entrada y salida: 
 Nombre del archivo donde quieres que escriba las concentraciones de los tipos de agua iniciales y externas?
 Nombre del archivo donde quieres que escriba las concentraciones despues de la mezcla reactiva?
 Nombre del archivo que contiene las concentraciones despues de resolver una iteracion de transporte conservativo? IMPORTANTE: El archivo debe estar en el directorio del problema, el numero de filas tiene que ser el numero de componentes y el numero de columnas el numero de targets.
 Paso de tiempo inicial?
 Paso de tiempo constante? (1: si, 0: no)
 Procedemos al bucle de mezcla reactiva. Tendras que actualizar el archivo con las concentraciones obtenidas despues de resolver el transporte conservativo en cada iteracion.


At line 34 of file interfaz_comps_arch.f90 (unit = 1, file = 'C:\Users\NRBRT\clone\JORDI\interfaz_remix-main1\examples\gypsum_eq\u_tilde.dat')
Fortran runtime error: End of file

Error termination. Backtrace:
#0  0xf085d649 in ???
#1  0xf07da6f1 in ???
#2  0xf06ed3fb in ???
#3  0xf07dbb9c in ???
#4  0xf087ca06 in ???
#5  0xf07df9ff in ???
#6  0xf0990795 in ???
#7  0xf07c6864 in ???
#8  0x30ca8127 in ???
#9  0x30cbab8f in ???
#10  0x30cbb0d9 in ???
#11  0x30b7133f in ???
#12  0x30b71145 in ???
#13  0x9df87613 in ???
#14  0x9ed626a0 in ???
#15  0xffffffff in ???


# 4. Utilizando conc_s.txt, se construye BTN y se ejecuta MT3D-USGS nv_veces. 
# La salida de MT3D-USGS se guarda en conc_i.m3d y se repite el ciclo: 
# (Ejecutar main_interfaz_local)(Construir BTN)(Ejecutar MT3D-USGS)



In [10]:
# Esta función genera un ciclo con los puntos 1) y 2).
# Ejecuta GWF-MT3D-NVG.ipynb haciendo 1): 
# 1) Lee conc_s.txt para modificar transport.btn y ejecutar GWF-MT3D-NVG.ipynb para generar conc_i.m3d.
# 2) Ejecuta RT.exe que lee conc_i.m3d, lo modifica y genera conc_s.txt
# 3) Se repite el ciclo [1),2)]
#******************************************************************************************************
# 1. Importación de librerías
import numpy as np
import subprocess
import time
import ipywidgets as widgets
from IPython.display import display

# 2. Rutas de archivos
#entrada_txt = "conc_s.txt"   # u_new.dat
#salida_m3d = "conc_i.m3d"    # u_tilde
entrada_txt = "u_new.dat"   # u_new.dat
salida_m3d = "u_tilde.dat"    # u_tilde
main_interfaz = "main_interfaz_local.exe" 
#ejecutable = "main_interfaz.exe" 

# 3. Slider para controlar nv
nv_slider = widgets.IntSlider(value=1, min=1, max=40, step=1, description="nv")
run_button = widgets.Button(description="Iniciar ciclo ")
output_area = widgets.Output()
contador_ejecuciones = 0

# 4. Función que ejecuta el ciclo
def ciclo_nv(b):
    global contador_ejecuciones
    output_area.clear_output()
    
    with output_area:
        contador_ejecuciones += 1
        print(f"🟢  Ejecución #{contador_ejecuciones}")        
        nv = nv_slider.value

        for i in range(nv):
            print(f"\n--- Iteración {i+1} de {nv} ---")
            
            #XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
            # EJECUTAR: (GWF,MT3D) ó GWF-MT3D-NVG
            # Condición según contador de ejecuciones
            if contador_ejecuciones == 1 and i == 0:
                print("▶️✅ Primera ejecución: corriendo GWF.ipynb y MT3D.ipynb...")
                %run GWF.ipynb  # Ejecutar GWF
                print(" 🟢🟢 Termina GWF, Inicia MT3D")
                %run MT3D.ipynb # Ejecutar MT3D
            else:
                print("▶️✅ Solo corriendo MT3D.ipynb...")
                %run MT3D.ipynb
                #%run GWF-MT3D-NVG.ipynb

            #XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
                # Ejecutar (main_interfaz_local.exe) (o reemplazar por WMA)
            try:
                print(f"✅ Ejecutando {main_interfaz} ...\n")
                proceso = subprocess.Popen([main_interfaz], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

                for linea in proceso.stdout:
                    print(linea, end="")  # imprime la salida línea por línea en tiempo real

                proceso.wait()
                if proceso.returncode != 0:
                    print(f"❌ El main_interfaz terminó con error. Código: {proceso.returncode}")
                    return  # o break, si estás dentro de un ciclo
            except Exception as e:
                print(f"❌ Error al ejecutar {main_interfaz}: {e}")
                return  # o break

                
            print("✅ Construimos BTN, ejecutamos MT3D-USGS...\n")
            print("⏸️ Pausa de 1 segundo...\n")
            time.sleep(1)

# 5. Conectar botón
run_button.on_click(ciclo_nv)

# 6. Mostrar interfaz
display(nv_slider, run_button, output_area)

IntSlider(value=1, description='nv', max=40, min=1)

Button(description='Iniciar ciclo ', style=ButtonStyle())

Output()